In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
import joblib


In [4]:
df_train=pd.read_csv("../data/train_clean.csv")
df_test=pd.read_csv("../data/test_clean.csv")
df_train.head()

,aaahhhhhhh scene,aaliyah,aaliyah small,aaron,aaron eckhart,aaron sheritt,aaron sheritts,aaron sorkin,ababy,ababy deer,...,zombie take,zombie title,zombie trap,zombie vomitte,zombie vs,zombie whistle,zombiefication,zombiefication party,zone,zoom
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
X_train_vec = df_train.drop(['label'], axis=1)
X_test_vec  = df_test.drop(['label'], axis=1)

y_train = df_train['label']
y_test  = df_test['label']

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")  

mlflow.set_experiment("Projet_nlp")  

with mlflow.start_run():

    model = MultinomialNB()
    param_grid = {'alpha': [0.1, 0.5, 1, 5]}  # valeurs à tester

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=5,
        scoring='f1_macro',
        n_jobs=-1
    )

    grid.fit(X_train_vec, y_train)

    # Log des résultats détaillés
    results = pd.DataFrame(grid.cv_results_)[['param_alpha', 'mean_test_score', 'std_test_score', 'rank_test_score']]
    print("Résultats détaillés de chaque alpha testé :")
    print(results)
    
    # Tu peux loguer chaque alpha et score
    for idx, row in results.iterrows():
        mlflow.log_metric(f"mean_test_score_alpha_{row['param_alpha']}", row['mean_test_score'])
        mlflow.log_metric(f"std_test_score_alpha_{row['param_alpha']}", row['std_test_score'])

    # Log du meilleur paramètre
    best_alpha = grid.best_params_['alpha']
    best_score = grid.best_score_
    print("_"*52)
    print("Meilleur alpha trouvé :", best_alpha)
    print("Meilleur score CV (F1-macro) :", best_score)
    
    mlflow.log_param("best_alpha", best_alpha)
    mlflow.log_metric("best_cv_f1_macro", best_score)

    # -------------------------------
    # 4️⃣ Évaluation sur le test
    # -------------------------------
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test_vec)

    acc = accuracy_score(y_test, y_pred)
    print("Accuracy sur test:", acc)
    print(classification_report(y_test, y_pred))
    
    mlflow.log_metric("test_accuracy", acc)
    
    # Log du modèle
    mlflow.sklearn.log_model(best_model, "naive_bayes_model")

    # Optionnel : log du DataFrame des résultats en CSV
    results.to_csv("grid_search_results.csv", index=False)
    mlflow.log_artifact("grid_search_results.csv")


2026/01/27 21:43:19 INFO mlflow.tracking.fluent: Experiment with name 'Projet_nlp' does not exist. Creating a new experiment.


Résultats détaillés de chaque alpha testé :
   param_alpha  mean_test_score  std_test_score  rank_test_score
0          0.1         0.781418        0.003349                1
1          0.5         0.776084        0.023850                2
2          1.0         0.739819        0.026489                3
3          5.0         0.610791        0.052482                4
____________________________________________________
Meilleur alpha trouvé : 0.1
Meilleur score CV (F1-macro) : 0.7814182184988755


2026/01/27 21:44:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy sur test: 0.735
              precision    recall  f1-score   support

           0       0.73      0.77      0.75       103
           1       0.74      0.70      0.72        97

    accuracy                           0.73       200
   macro avg       0.74      0.73      0.73       200
weighted avg       0.74      0.73      0.73       200

🏃 View run marvelous-roo-188 at: http://127.0.0.1:5000/#/experiments/1/runs/06bb8d0e0f004071b69acdfc4f25b3e3
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [ ]:
# Sauvegarde locale avec joblib en .pkl
joblib.dump(best_model, "../model/naive_bayes_best_model.pkl")
print("Modèle sauvegardé localement dans naive_bayes_best_model.pkl")


NameError: name 'best_model' is not defined